# 2. Reconstruction

Reconstruct the included static-phantom k-space data using algebraic dead-time infilling, ZTE-STAR with temporal TV, and conventional radial gridding.

Run **1_trajectory.ipynb** first. Notebook 2 requires an NVIDIA GPU with CUDA.


In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add the repository root to the import path when the notebook is launched
# directly from the repository rather than from an installed package.
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import config, reconstruction

cfg = config.get()
data_dir = config.DATA_DIR / "simulated"
traj_dir = config.OUTPUT_DIR / "trajectory"

# Load the included static-phantom k-space and the paired trajectories created
# by the first notebook.
ksp_spiral = np.load(data_dir / "ksp_spiral.npy")
ksp_radial = np.load(data_dir / "ksp_radial.npy")
phantom = np.load(data_dir / "phantom.npy")
labels = np.load(data_dir / "phantom_labels.npy")
spiral_pair = np.load(
    traj_dir / "spiral_phyllotaxis_pairspoke.npy"
)
radial_pair = np.load(
    traj_dir / "bruker_radial_pairspoke.npy"
)

print("spiral:", ksp_spiral.shape)
print("radial:", ksp_radial.shape)


## Dead-time infilling


In [ ]:
# Apply the same paired-spoke dead-time infilling to both acquisitions
# before reconstruction.
start_time = time.time()
data_spiral = reconstruction.prepare_pairspoke_data(
    ksp_spiral,
    n_adc_delay=cfg.n_adc_delay,
    n_seg=cfg.M,
    n_gap=cfg.n_gap,
    over_sampling=cfg.over_sampling,
)
data_radial = reconstruction.prepare_radial_repetitions(
    ksp_radial,
    n_rep=cfg.radial_n_rep,
    n_pro=cfg.radial_n_pro,
    n_adc_delay=cfg.n_adc_delay,
    n_gap=cfg.n_gap,
    over_sampling=cfg.over_sampling,
)
print(f"Dead-time infilling: {time.time() - start_time:.1f} s")

# Plot a small neighborhood around k-space center to show which paired-spoke
# samples were restored by the algebraic infilling step.
center = data_spiral.shape[1] // 2
half_width = 4 * cfg.n_gap
segment = np.abs(
    data_spiral[
        0,
        center - half_width:center + half_width,
        0,
        0,
    ]
)
sample_index = np.arange(
    center - half_width,
    center + half_width,
)
gap = np.abs(sample_index - center) < cfg.n_gap

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(
    sample_index[~gap],
    segment[~gap],
    "o",
    label="measured",
)
ax.plot(
    sample_index[gap],
    segment[gap],
    "o",
    label="infilled",
)
ax.plot(
    sample_index,
    segment,
    "-",
    lw=1,
    alpha=0.5,
)
ax.set_xlabel("sample along paired spoke")
ax.set_ylabel("|k-space|")
ax.set_title("Dead-time gap after algebraic infilling")
ax.legend()
plt.tight_layout()


## Reconstruction


In [ ]:
out = config.OUTPUT_DIR / "reconstruction"
(out / "spiral").mkdir(parents=True, exist_ok=True)
(out / "radial").mkdir(parents=True, exist_ok=True)

# Reconstruct the ZTE-STAR data with the TTV model.
start_time = time.time()
ttv, _ = reconstruction.run_ttv_spiral(
    data_spiral,
    spiral_pair,
    M=cfg.M,
    P=cfg.P,
    n_window=cfg.n_window,
    lamda=cfg.lamda,
    n_iter=cfg.n_iter,
    matrix_size=cfg.matrix_size,
    n_calc_seg=cfg.n_calc_seg,
    finite_support=cfg.finite_support,
    save_root=out / "spiral",
)
print(
    f"TTV reconstruction: {time.time() - start_time:.1f} s"
)

# Reconstruct the conventional radial data by density-compensated gridding.
start_time = time.time()
radial = reconstruction.run_gridding(
    data_radial,
    radial_pair,
    n_window=cfg.radial_n_window,
    matrix_size=cfg.matrix_size,
    n_calc_seg=cfg.n_calc_seg,
    finite_support=cfg.finite_support,
    save_root=out / "radial",
)
print(f"Radial gridding: {time.time() - start_time:.1f} s")


## Result


In [ ]:
def match_scale(volume):
    """Match a reconstructed volume to the phantom mean within the host sphere.

    Args:
        volume: Complex-valued or real-valued reconstructed volume.

    Returns:
        Magnitude image rescaled so that label 1 has the same mean intensity
        as the reference phantom.
    """
    magnitude = np.abs(volume)
    host_sphere = labels == 1
    scale = (
        phantom[host_sphere].mean()
        / magnitude[host_sphere].mean()
    )
    return magnitude * scale


# Display the central slice with a common intensity range so visual differences
# reflect reconstruction quality rather than independent windowing.
z_index = cfg.L // 2
ttv_img = match_scale(ttv[0])
radial_img = match_scale(radial[0])
vmax = float(phantom.max()) * 1.1

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, title, image in zip(
    axes,
    [
        "Reference phantom",
        "ZTE-STAR + TTV",
        "Conventional radial",
    ],
    [phantom, ttv_img, radial_img],
):
    ax.imshow(
        image[z_index],
        cmap="gray",
        vmin=0,
        vmax=vmax,
    )
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
